In [5]:
import pandas as pd
import glob
import os

# Process NDVI data

In [6]:
source_dir = './dataset/ndvi'
pattern = os.path.join(source_dir, "NDVI_LUX_*.csv")
files = sorted(glob.glob(pattern))

if not files:
    raise FileNotFoundError(f"No files matched {pattern}")
    
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# Group by Year and Month, compute median and max of NDVI
grouped = df.groupby(['Year', 'Month'])['NDVI']
median_ndvi = grouped.median().reset_index(name='Median')
max_ndvi = grouped.max().reset_index(name='Max')

# Save results
median_out = os.path.join(source_dir, "monthly_median_ndvi_2017_2024.csv")
max_out = os.path.join(source_dir, "monthly_max_ndvi_2017_2024.csv")

median_ndvi.to_csv(median_out,  index=False)
max_ndvi.to_csv(max_out, index=False)

print(f'Median NDVI:\n {median_ndvi.head()}')
print(f'Max NDVI:\n {max_ndvi.head()}')

Median NDVI:
    Year  Month    Median
0  2017      1  0.074462
1  2017      2  0.363243
2  2017      3  0.587939
3  2017      4  0.460918
4  2017      5  0.680700
Max NDVI:
    Year  Month       Max
0  2017      1  0.074462
1  2017      2  0.462868
2  2017      3  0.587939
3  2017      4  0.589868
4  2017      5  0.722713


# Aggregate Daily Weather Data from NASA POWER

In [10]:
import pandas as pd

# 1. load & parse dates
df = pd.read_csv("./dataset/weather/POWER_Point_Daily_20000101_20241231_049d82N_006d13E_LST.csv", skiprows=18)
df['Date']  = pd.to_datetime(df['YEAR']*1000 + df['DOY'], format='%Y%j')
# Ex: YEAR=2000, DOY = 32: 2000 * 1000 + 32  = 2000000 + 32  = 2000032
# %Y means “four‐digit year” (e.g. 2000)
# %j means “three‐digit day-of-year” (e.g. 032 for February 1)
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# 1a. convert cloud from percent (62.3) to fraction (0.623)
df['CLOUD_AMT'] = df['CLOUD_AMT'] / 100.0

# 2. define aggregations
agg_map = {
    'PRECTOTCORR': 'sum',
    'QV2M'       : 'mean',
    'T2M'        : 'mean',
    'T2M_MIN'    : 'min',
    'T2M_MAX'    : 'max',
    'PS'         : 'mean',
    'WS2M'       : 'mean',
    'RH2M'       : 'mean',
    'T2MDEW'     : 'mean',
    'CLOUD_AMT'  : 'mean',
}

monthly = (
    df
    .groupby(['Year','Month'])
    .agg(agg_map)
    .reset_index()
)

# 3. human‐friendly names
rename_map = {
    'PRECTOTCORR': 'Precipitation (mm)',
    'QV2M'       : 'Specific Humidity (g/kg)',
    'T2M'        : 'Avg. Temp (C)',
    'T2M_MIN'    : 'Min Temp (C)',
    'T2M_MAX'    : 'Max Temp (C)',
    'PS'         : 'Surface Pressure (kPa)',
    'WS2M'       : 'Wind Speed (m/s)',
    'RH2M'       : 'Relative Humidity (%)',
    'T2MDEW'     : 'Dew Point (C)',
    'CLOUD_AMT'  : 'Cloud Amount',
}

monthly = monthly.rename(columns=rename_map)

# 4. save
monthly.to_csv("./dataset/weather/monthly_weather_data.csv", index=False)

monthly.head()


,Year,Month,Precipitation (mm),Specific Humidity (g/kg),Avg. Temp (C),Min Temp (C),Max Temp (C),Surface Pressure (kPa),Wind Speed (m/s),Relative Humidity (%),Dew Point (C),Cloud Amount
0,2000,1,52.99,3.809355,-0.036774,-10.24,7.77,97.526452,3.297419,95.357097,-0.767097,0.600419
1,2000,2,95.37,4.368966,2.313448,-3.62,9.78,97.167241,3.937241,93.090345,1.263103,0.736448
2,2000,3,78.81,4.862903,4.362581,-3.59,13.45,97.092903,3.499032,89.878387,2.740968,0.781387
3,2000,4,52.20,6.022667,8.017000,-3.09,20.87,96.041000,2.996333,84.474667,5.356000,0.749733
4,2000,5,78.89,8.006129,13.357419,2.96,24.32,96.808387,2.599677,80.557097,9.844194,0.795097


In [11]:
df['Date'].head()

0   2000-01-01
1   2000-01-02
2   2000-01-03
3   2000-01-04
4   2000-01-05
Name: Date, dtype: datetime64[ns]

# Merge NDVI and weather data for 2017-2024 to perform NDVI prediction

In [4]:
ndvi_df = pd.read_csv('./dataset/ndvi/monthly_median_ndvi_2017_2024.csv')
weather_df = pd.read_csv('./dataset/weather/monthly_weather_data.csv')

#Filter weather to the NDVI period (2017-2014)
weather_df = weather_df[(weather_df['Year'] >=2017) & (weather_df['Year'] <= 2024)]

# Merge on Year and Month
merged_df = pd.merge(ndvi_df, weather_df, on=['Year', 'Month'], how='inner')
merged_df.rename(columns={'Median': 'NDVI'}, inplace=True)

print(merged_df.head())

os.makedirs('./dataset/ndvi_prediction', exist_ok=True)
merged_df.to_csv('./dataset/ndvi_prediction/monthly_ndvi_weather_2017_2024.csv', index=False)

   Year  Month      NDVI  Precipitation (mm)  Specific Humidity (g/kg)  \
0  2017      1  0.074462               44.53                  2.856129   
1  2017      2  0.363243               58.09                  4.484643   
2  2017      3  0.587939               52.41                  5.312258   
3  2017      4  0.460918               13.32                  4.828333   
4  2017      5  0.680700               52.99                  7.751935   

   Avg. Temp (C)  Min Temp (C)  Max Temp (C)  Surface Pressure (kPa)  \
0      -3.808710        -13.95          2.87               97.338387   
1       2.467500         -3.02          9.83               96.763571   
2       6.326774         -1.68         18.25               96.825161   
3       6.524333         -3.38         19.01               97.221333   
4      13.457742          0.37         28.16               96.923548   

   Wind Speed (m/s)  Relative Humidity (%)  Dew Point (C)  Cloud Amount  
0          2.950968              94.858387      